# Talking Avatar Generator — Wan2GP + Edge-TTS

Generate a lip-synced talking head video from a **photo** and a **script**.

**Pipeline:**
1. Write your script → generate speech audio (Edge-TTS, free, no API key)
2. Upload your photo → animate it with Wan2GP (lip-sync + facial expressions)
3. Download the result

**Requirements:**
- Google account with Colab access
- Runtime → Change runtime type → **GPU** (free T4 works)
- A photo of a face (square crop, face forward, 50-70% of frame)

> **Free tier note:** T4 GPU (15GB VRAM) fits the `Wan 2.2 FastWan 5B` model at 480p. For higher quality, upgrade to Colab Pro.

---
## Step 1 — Confirm GPU is available

If this fails, go to **Runtime → Change runtime type → GPU** and try again.

In [ ]:
import subprocess, torch

try:
    result = subprocess.run(['nvidia-smi'], check=True, capture_output=True, text=True)
    print(result.stdout)
except Exception as exc:
    raise RuntimeError(
        'GPU not detected. Go to Runtime → Change runtime type → select GPU → Save → rerun this cell.'
    ) from exc

print(f'PyTorch CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name: {torch.cuda.get_device_name(0)}')
    vram_gb = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    print(f'VRAM: {vram_gb:.1f} GB')
    if vram_gb < 12:
        print('WARNING: Low VRAM. Stick to 480p resolution in the Gradio UI.')

---
## Step 2 — Install Edge-TTS (text-to-speech)

Edge-TTS converts your script into natural-sounding speech. Free, no API key, 300+ voices.

In [ ]:
!pip install edge-tts -q
print('Edge-TTS installed.')

---
## Step 3 — List available English voices

Run this to see all English voices. Pick one for Step 4.

**Recommended male voices:**
- `en-US-GuyNeural` — natural, conversational
- `en-US-ChristopherNeural` — deep, authoritative
- `en-GB-RyanNeural` — British accent

**Recommended female voices:**
- `en-US-JennyNeural` — clear, professional
- `en-US-AriaNeural` — expressive
- `en-GB-SoniaNeural` — British accent

In [ ]:
import edge_tts, asyncio

voices = await edge_tts.list_voices()
en = [v for v in voices if v['Locale'].startswith('en-')]
print(f'{len(en)} English voices available. Using en-US-ChristopherNeural.')
print('Voice: deep, authoritative tone - perfect for hype scripts.')

---
## Step 4 — Write your script and generate speech

Edit the `script` variable below with your text. Then run the cell to generate audio.

**Tips:**
- Keep scripts under 30 seconds for free Colab (shorter = faster generation)
- Use natural punctuation: commas, periods, exclamation marks
- Adjust `rate` for speed: `+0%` = normal, `-20%` = slower, `+20%` = faster
- Adjust `pitch` for tone: `+0Hz` = normal, `-10Hz` = deeper, `+10Hz` = higher

In [ ]:
import edge_tts, asyncio, os, subprocess
from IPython.display import Audio, display

voice = "en-US-ChristopherNeural"
rate = "+5%"
pitch = "-2Hz"

segments = {
    'seg1a': 'Listen up, MYTHOS just dropped, and everybody out here is acting like it is just another model. BRO, WAKE UP. This thing is not an upgrade. It is a detonation.',
    'seg1b': 'Mythos is the first AI that does not wait for instructions. It hunts for outcomes. You give it a goal, it builds the plan, the steps, the strategy, the execution.',
    'seg1c': 'It is like hiring a Navy SEAL, a CEO, and a supercomputer, for the price of a coffee.',
    'seg2': 'People are scared of it. GOOD. Be scared. Because Mythos is the model that separates the spectators from the dominators.',
    'seg3': 'Everyone else is playing with chatbots. I am telling you, Mythos is a business weapon. You want to 10X your output? You want to dominate your niche? You want to stop being a consumer and start being a MACHINE?',
    'seg4': 'Then stop sleeping on Mythos. This is the moment the whole AI game flips. The ones who move NOW, they are the ones who own the next decade. Get in. Or get left behind.'
}

os.makedirs('/content/audio', exist_ok=True)
print(f'Generating {len(segments)} audio segments...\n')

for name, script in segments.items():
    wav = f'/content/audio/{name}.wav'
    mp3 = f'/content/audio/{name}.mp3'
    comm = edge_tts.Communicate(script, voice, rate=rate, pitch=pitch)
    await comm.save(mp3)
    subprocess.run(['ffmpeg', '-i', mp3, '-ar', '16000', '-ac', '1', wav, '-y'], capture_output=True)
    r = subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
                       '-of', 'default=noprint_wrappers=1:nokey=1', wav],
                      capture_output=True, text=True)
    dur = float(r.stdout.strip()) if r.stdout.strip() else 0
    print(f'  {name}: {dur:.1f}s')

print(f'\nAll {len(segments)} segments saved to /content/audio/')
print('Files: seg1a.wav, seg1b.wav, seg1c.wav, seg2.wav, seg3.wav, seg4.wav')

---
## Step 5 — Upload your photo

**Photo requirements for best results:**
- **Square crop** — your face should be roughly centered
- **Face forward** — looking at the camera, rotation < 30 degrees
- **Face size** — 50-70% of the image frame
- **Good lighting** — even lighting, no harsh shadows on face
- **Clean background** — simple background works best
- **Resolution** — at least 512x512 pixels

Run the cell and use the file picker to upload your photo.

In [ ]:
from google.colab import files
import os

print('Upload your photo (face forward, good lighting):')
uploaded = files.upload()

if not uploaded:
    raise RuntimeError('No file uploaded. Run this cell again.')

PHOTO_PATH = os.path.join('/content', list(uploaded.keys())[0])

from PIL import Image
img = Image.open(PHOTO_PATH)
print(f'\nPhoto saved: {PHOTO_PATH}')
print(f'Size: {img.size[0]}x{img.size[1]}')
if img.size[0] != img.size[1]:
    print('TIP: Crop to square for best results.')
display(img)

---
## Step 6 — Set up Wan2GP

This clones the Wan2GP repository and installs all dependencies. Takes ~5-10 minutes on first run.

**Tip:** Set `USE_GOOGLE_DRIVE = True` if you want to keep model downloads across sessions.

In [ ]:
#@markdown ### Workspace Configuration
#@markdown Set to True to persist models/checkpoints across Colab sessions via Google Drive.
USE_GOOGLE_DRIVE = False  #@param {type:"boolean"}

from pathlib import Path

DRIVE_MOUNT_POINT = Path('/content/drive')
WAN2GP_ROOT = Path('/content/Wan2GP').resolve()
EPHEMERAL_DATA_ROOT = Path('/content/Wan2GP-data').resolve()
PERSISTENT_DATA_ROOT = (DRIVE_MOUNT_POINT / 'MyDrive' / 'Wan2GP-data').resolve()

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount(str(DRIVE_MOUNT_POINT), force_remount=False)
    WAN_DATA_ROOT = PERSISTENT_DATA_ROOT
    data_mode = 'Google Drive (persistent)'
else:
    WAN_DATA_ROOT = EPHEMERAL_DATA_ROOT
    data_mode = 'Colab runtime (ephemeral)'

WAN_CKPTS_DIR = (WAN_DATA_ROOT / 'ckpts').resolve()
WAN_LORAS_DIR = (WAN_DATA_ROOT / 'loras').resolve()
WAN_OUTPUTS_DIR = (WAN_DATA_ROOT / 'outputs').resolve()
WAN_CACHE_DIR = (WAN_DATA_ROOT / 'cache').resolve()
WAN_LTX2_LORAS_DIR = (WAN_LORAS_DIR / 'ltx2').resolve()
WAN_LTX2_22B_LORAS_DIR = (WAN_LORAS_DIR / 'ltx2_22B').resolve()

WAN2GP_ROOT.parent.mkdir(parents=True, exist_ok=True)
WAN_DATA_ROOT.mkdir(parents=True, exist_ok=True)
WAN_CKPTS_DIR.mkdir(parents=True, exist_ok=True)
WAN_LORAS_DIR.mkdir(parents=True, exist_ok=True)
WAN_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
WAN_CACHE_DIR.mkdir(parents=True, exist_ok=True)
WAN_LTX2_LORAS_DIR.mkdir(parents=True, exist_ok=True)
WAN_LTX2_22B_LORAS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Storage mode: {data_mode}')
print(f'Wan2GP root: {WAN2GP_ROOT}')
print(f'Outputs dir: {WAN_OUTPUTS_DIR}')

In [ ]:
import shutil, subprocess
from pathlib import Path

def safe_link(repo_path: Path, data_path: Path) -> None:
    """Safely link a repo subdirectory to the data directory."""
    data_path.mkdir(parents=True, exist_ok=True)
    try:
        # Already a correct symlink
        if repo_path.is_symlink() and repo_path.resolve() == data_path.resolve():
            return
        # Symlink points elsewhere — remove it
        if repo_path.is_symlink():
            repo_path.unlink()
        # Real directory with content — move contents then remove
        if repo_path.is_dir():
            for child in list(repo_path.iterdir()):
                dest = data_path / child.name
                if not dest.exists():
                    shutil.move(str(child), str(dest))
            try:
                repo_path.rmdir()
            except OSError:
                pass  # Directory not empty, skip
        # Create the symlink
        repo_path.symlink_to(data_path, target_is_directory=True)
        print(f'  Linked {repo_path.name} -> {data_path}')
    except Exception as e:
        print(f'  WARNING: Could not link {repo_path.name}: {e}')
        print(f'  Continuing without symlink — you may need to manually set paths.')

repo_url = 'https://github.com/deepbeepmeep/Wan2GP.git'
if WAN2GP_ROOT.exists():
    try:
        status = subprocess.run(
            ['git', '-C', str(WAN2GP_ROOT), 'status', '--porcelain'],
            check=True, capture_output=True, text=True,
        )
        if status.stdout.strip():
            print('Repository exists with local changes. Skipping pull.')
        else:
            print('Pulling latest Wan2GP changes...')
            subprocess.run(['git', '-C', str(WAN2GP_ROOT), 'pull'], check=True)
    except Exception as e:
        print(f'Git pull failed: {e}. Using existing repo.')
else:
    print('Cloning Wan2GP repository...')
    subprocess.run(['git', 'clone', repo_url, str(WAN2GP_ROOT)], check=True)

print('Setting up data directories...')
safe_link(WAN2GP_ROOT / 'ckpts', WAN_CKPTS_DIR)
safe_link(WAN2GP_ROOT / 'loras', WAN_LORAS_DIR)
safe_link(WAN2GP_ROOT / 'outputs', WAN_OUTPUTS_DIR)
print('Wan2GP repository ready.')

In [ ]:
import os, subprocess
env = os.environ.copy()
env['DEBIAN_FRONTEND'] = 'noninteractive'
subprocess.run(['sudo', 'apt-get', 'update', '-qq'], check=False, env=env)
subprocess.run(['sudo', 'apt-get', 'install', '-y', '--no-install-recommends',
    'ffmpeg', 'libglib2.0-0', 'libgl1', 'libportaudio2'], check=False, env=env)
print('System deps installed.')

In [ ]:
import importlib, os, subprocess, sys
env = os.environ.copy()

# Check existing torch
torch_ok = False
try:
    import torch
    if torch.cuda.is_available():
        print(f'CUDA torch ready: {torch.__version__} (CUDA {torch.version.cuda})')
        torch_ok = True
except ImportError:
    pass

if not torch_ok:
    print('Installing torch with CUDA...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'], check=False, env=env)
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch', 'torchvision', 'torchaudio',
                        '--index-url', 'https://download.pytorch.org/whl/cu128'],
                       capture_output=True, text=True, env=env)
    if r.returncode != 0:
        print(f'Warning: {r.stderr[-200:]}')

# xformers optional
try:
    import xformers
    print(f'xformers: {xformers.__version__}')
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'xformers',
                    '--index-url', 'https://download.pytorch.org/whl/cu128'],
                   capture_output=True, env=env)

# Wan2GP requirements
req = WAN2GP_ROOT / 'requirements.txt'
if req.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(req)], check=False, env=env)
    print('Wan2GP requirements installed.')

import torch
print(f'\nTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    print(f'VRAM: {vram:.1f} GB')

In [ ]:
# Fix matplotlib backend for headless Colab
from pathlib import Path
target = WAN2GP_ROOT / 'preprocessing/matanyone/tools/interact_tools.py'
if target.exists():
    text = target.read_text()
    if "matplotlib.use('TkAgg')" in text:
        target.write_text(text.replace("matplotlib.use('TkAgg')", "matplotlib.use('Agg')", 1))
        print('Fixed matplotlib backend for Colab.')
    else:
        print('Matplotlib backend already OK.')
else:
    print('interact_tools.py not found, skipping.')

---
## Step 7 — Launch Wan2GP Gradio UI

Run this cell to start the Gradio web interface. A link will appear in the output — click it to open the UI.

**Keep this cell running** while you use the UI. Stop it with the square button when done.

In [ ]:
import os, subprocess, sys, threading, time
from pathlib import Path

wgp_py = WAN2GP_ROOT / 'wgp.py'
if not wgp_py.exists():
    found = list(WAN2GP_ROOT.rglob('wgp.py'))
    if found:
        wgp_py = found[0]
    else:
        raise RuntimeError(f'wgp.py not found. Re-run Step 6.')

import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Runtime -> Change runtime type -> GPU.')

env = os.environ.copy()
env['WAN_CACHE_DIR'] = str(WAN_CACHE_DIR)
env['HF_HOME'] = str(WAN_CACHE_DIR / 'huggingface')
env['HUGGINGFACE_HUB_CACHE'] = str(WAN_CACHE_DIR / 'huggingface' / 'hub')
env['TRANSFORMERS_CACHE'] = str(WAN_CACHE_DIR / 'huggingface' / 'transformers')
env['TORCH_HOME'] = str(WAN_CACHE_DIR / 'torch')
env['XDG_CACHE_HOME'] = str(WAN_CACHE_DIR / '.cache')

cmd = [sys.executable, '-u', str(wgp_py), '--listen', '--server-port', '7860', '--share', '--profile', '5']

print('='*60)
print('WAN2GP GRADIO UI')
print('='*60)
print('1. Wait for the public URL below')
print('2. Click the URL to open Gradio')
print('3. Model: Wan 2.2 TextImage2Video 5B FastWan')
print('4. Upload photo from Step 5')
print('5. Upload audio: /content/audio/seg1a.wav (first video)')
print('6. Resolution: 480p | Frames: 81 | Generate!')
print('7. After download, re-run this cell with next .wav file')
print('='*60 + '\n')

process = subprocess.Popen(cmd, cwd=str(WAN2GP_ROOT), env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

stop_event = threading.Event()
def keepalive():
    while not stop_event.is_set():
        time.sleep(45)
        if not stop_event.is_set():
            print('[keepalive] Still running...')
t = threading.Thread(target=keepalive, daemon=True)
t.start()

try:
    for line in iter(process.stdout.readline, ''):
        if not line: break
        print(line, end='')
except KeyboardInterrupt:
    print('\nStopping...')
    process.terminate()
finally:
    stop_event.set()
    process.wait()
    print(f'Stopped (exit code: {process.returncode}).')

---
## Step 8 — Gradio UI Walkthrough (Talking Head)

Once the Gradio UI opens in your browser, follow these steps:

### 8a. Select the model
- Look for the **Model** dropdown at the top
- Select: **`Wan 2.2 TextImage2Video 5B FastWan`**
- This is the only model that fits in 15GB VRAM on free Colab

### 8b. Upload your photo
- Find the **Image** upload area
- Upload the same photo from Step 5

### 8c. Upload your audio
- Find the **Audio** upload area (may be labeled "Driving Audio" or "Audio Input")
- Upload the `speech.wav` file from Step 4
- You can download it from Colab files panel (folder icon on the left)

### 8d. Set generation parameters
- **Resolution:** 480p (recommended for free Colab)
- **Frames:** 81 (about 5 seconds at 16fps)
- **Guidance Scale:** 5.0-7.0
- **Steps:** 20-30

### 8e. Generate
- Click **Generate** or **Run**
- Wait 3-8 minutes for the video to render
- The result will appear in the output area

### 8f. Download
- Right-click the output video → Save
- Or check the Wan2GP outputs folder in the Colab file browser

---
## Step 9 — Download your video

Run this cell to find and download your generated video.

In [ ]:
import glob, os
from pathlib import Path
from google.colab import files

try:
    _ = WAN_OUTPUTS_DIR
except NameError:
    WAN_OUTPUTS_DIR = Path('/content/Wan2GP-data/outputs')
try:
    _ = WAN2GP_ROOT
except NameError:
    WAN2GP_ROOT = Path('/content/Wan2GP')

search = [
    str(WAN_OUTPUTS_DIR / '**/*.mp4'),
    str(WAN2GP_ROOT / 'outputs' / '**/*.mp4'),
    '/content/**/*.mp4',
]
videos = []
for p in search:
    videos.extend(glob.glob(p, recursive=True))
videos = sorted(set(videos), key=os.path.getmtime, reverse=True)

if not videos:
    print('No videos yet. Generate one in Gradio first.')
else:
    print(f'Found {len(videos)} video(s):\n')
    for i, v in enumerate(videos, 1):
        mb = os.path.getsize(v) / (1024*1024)
        print(f'{i}. {v} ({mb:.1f} MB)')
    latest = videos[0]
    print(f'\nDownloading: {latest}')
    files.download(latest)

---
## Tips for Better Results

### Photo tips
- Use a high-resolution photo (1024x1024 or larger)
- Face should be clearly visible, not obscured by hair or accessories
- Neutral or slight smile works best
- Avoid extreme angles or side profiles

### Audio tips
- Clear speech without background noise
- English only (Hallo2 was trained on English)
- Moderate speaking pace sounds most natural
- Add pauses with `...` or `-` in your script

### Generation tips
- Shorter clips (5-10 seconds) generate faster and more reliably
- If the mouth looks wrong, try a different guidance scale (5-7)
- Generate a few times — results vary with different seeds
- 480p is the sweet spot for free Colab T4

### Voice cloning (advanced)
- Wan2GP supports voice cloning via SeedVC
- Upload a voice sample in the Audio tab → Advanced → Voice Cloning
- Works with any video model output

---
## Troubleshooting

| Problem | Solution |
|---------|----------|
| GPU not detected | Runtime → Change runtime type → GPU |
| Out of memory | Lower resolution to 480p, reduce frames to 81 |
| Gradio link not appearing | Wait 2-3 minutes, check cell output for errors |
| Mouth not syncing well | Try different guidance scale (5-7), ensure audio is clear |
| Model download slow | Set `USE_GOOGLE_DRIVE = True` to cache models |
| Session disconnected | Re-run cells from Step 1 (models may re-download) |
| Video is blurry | Normal for 480p; try Colab Pro for 720p/1080p |